[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abhayraj01op/AI-Post-Studio/blob/main/AI_Post_Studio_Colab.ipynb)

# AI Post Studio — ComfyUI + Google Colab
Images → Posts → Biodata (Apna Connection).

**Kaise chalayein:** Upar menu mein *Runtime → Change runtime type → T4 GPU* chunein, phir *Runtime → Run all*. Aakhri cell ek `gradio.live` link dega — us par click karke app kholein.

Sab kuch (ComfyUI, models, photos, posts, biodata) aapki Google Drive ke `MyDrive/AI_Post_Studio` folder mein save hota hai.

In [ ]:
#@title 1. Settings
GITHUB_REPO = "https://github.com/abhayraj01op/AI-Post-Studio"  #@param {type:"string"}
TEXT_MODEL = "Qwen/Qwen2.5-7B-Instruct"  #@param ["Qwen/Qwen2.5-7B-Instruct", "Qwen/Qwen2.5-3B-Instruct"]
CHECKPOINT_URL = "https://huggingface.co/SG161222/RealVisXL_V5.0/resolve/main/RealVisXL_V5.0_fp16.safetensors"  #@param {type:"string"}
BASE = "/content/drive/MyDrive/AI_Post_Studio"


In [ ]:
#@title 2. Google Drive mount
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs(BASE, exist_ok=True)
print('Drive ready:', BASE)

In [ ]:
#@title 3. One-click setup (pehli baar ~10-15 min, baad mein ~2 min)
import os, subprocess
def sh(c):
    print('$', c); r = subprocess.run(c, shell=True)
    if r.returncode: raise SystemExit(f'Fail: {c}')
sh('apt-get -qq update && apt-get -qq install -y libraqm0 libfribidi0 aria2 > /dev/null')
# App code (har baar GitHub se latest)
if os.path.exists('/content/aps'): sh('cd /content/aps && git pull -q')
else: sh(f'git clone -q {GITHUB_REPO} /content/aps')
# ComfyUI Drive mein (ek baar install, hamesha save)
comfy = f'{BASE}/ComfyUI'
if not os.path.exists(f'{comfy}/main.py'): sh(f'git clone -q https://github.com/comfyanonymous/ComfyUI "{comfy}"')
else: sh(f'cd "{comfy}" && git pull -q')
sh(f'pip -q install -r "{comfy}/requirements.txt"')
sh('pip -q install -r /content/aps/requirements.txt')
# Photo model (Drive mein ek baar download)
ck = f'{comfy}/models/checkpoints'; os.makedirs(ck, exist_ok=True)
name = CHECKPOINT_URL.split('/')[-1].split('?')[0]
if not os.path.exists(f'{ck}/{name}'): sh(f'aria2c -q -x 16 -s 16 -d "{ck}" -o "{name}" "{CHECKPOINT_URL}"')
# Hindi fonts
fd = f'{BASE}/fonts'; os.makedirs(fd, exist_ok=True)
for w in ['ExtraBold','Bold','Medium']:
    if not os.path.exists(f'{fd}/Mukta-{w}.ttf'):
        sh(f'wget -q -O "{fd}/Mukta-{w}.ttf" https://github.com/google/fonts/raw/main/ofl/mukta/Mukta-{w}.ttf')
os.environ.update(APS_BASE=BASE, APS_CHECKPOINT=name, APS_TEXT_MODEL=TEXT_MODEL, HF_HOME=f'{BASE}/hf_cache')
print('Setup poora ho gaya ✔')

In [ ]:
#@title 4. Name test (4 din wala rule)
!cd /content/aps && python -m pytest -q tests

In [ ]:
#@title 5. Start — ComfyUI + App (link neeche aayega)
import subprocess, time, urllib.request, os
log = open('/content/comfyui.log', 'w')
subprocess.Popen(['python', 'main.py', '--listen', '127.0.0.1', '--port', '8188',
                  '--output-directory', f'{BASE}/data/comfy_output'], cwd=f'{BASE}/ComfyUI', stdout=log, stderr=log)
for _ in range(120):
    try: urllib.request.urlopen('http://127.0.0.1:8188/system_stats', timeout=2); print('ComfyUI chal raha ✔'); break
    except Exception: time.sleep(2)
else: print('ComfyUI start nahi hua, /content/comfyui.log dekhein')
!cd /content/aps && python -m app.main

**Tips**
- Colab band hone par bhi sab Drive mein safe hai. Agli baar bas *Run all*.
- ComfyUI ka apna UI bhi chahiye? `ComfyUI` ko `--listen 0.0.0.0` se chalakar cloudflared tunnel use karein (optional).
- Free Colab GPU kuch ghante baad band ho sakta hai; Colab Pro se lamba chalta hai.